In [1]:
# Cell 01 | Load course lesson pages
# Purpose: Download the official LLM Zoomcamp lesson Markdown files
#          from the fixed homework commit and parse them into documents.
# Key points: Keep only Markdown files under /lessons/ so every document
#             represents one course lesson page.
# Execution: Run this cell once. It downloads repository data from GitHub.

from gitsource import GithubRepositoryDataReader


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)


print("Lesson pages:", len(documents))
print("First document keys:", documents[0].keys())
print("First filename:", documents[0]["filename"])

Lesson pages: 72
First document keys: dict_keys(['content', 'filename'])
First filename: 01-agentic-rag/lessons/01-intro.md


In [2]:
# Cell 02 | Build and query the lesson search index
# Purpose: Index the course lesson documents with MinSearch and verify retrieval.
# Key points: Use content as the searchable text field and filename as metadata.
# Execution: Run after Cell 01. This cell performs local retrieval only;
#            no OpenAI API request is made.

import minsearch


index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query=query,
    num_results=5,
)

print("Top result:", results[0]["filename"])

print("\nTop 5 results:")
for rank, result in enumerate(results, start=1):
    print(f"{rank}. {result['filename']}")

Top result: 01-agentic-rag/lessons/14-agentic-loop.md

Top 5 results:
1. 01-agentic-rag/lessons/14-agentic-loop.md
2. 01-agentic-rag/lessons/15-frameworks.md
3. 01-agentic-rag/lessons/13-function-calling.md
4. 01-agentic-rag/lessons/11-agents-intro.md
5. 01-agentic-rag/lessons/16-other-frameworks.md


In [3]:
# Cell 03 | Define a homework-specific RAG helper
# Purpose: Adapt the RAG flow to lesson documents with filename/content fields.
# Key points: Keep the main project RAGBase unchanged and expose API usage
#             so Q3 can measure the number of input tokens.
# Execution: This cell only defines the helper and creates the client.
#            It does not send an OpenAI API request.

from dotenv import load_dotenv
from openai import OpenAI


load_dotenv("../.env")

client = OpenAI()


INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
""".strip()


PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


class LessonRAG:
    """RAG helper for lesson documents with filename/content fields."""

    def __init__(self, index, llm_client, model="gpt-5.4-mini"):
        self.index = index
        self.llm_client = llm_client
        self.model = model

    def search(self, query, num_results=5):
        """Retrieve the most relevant lesson documents."""
        return self.index.search(
            query=query,
            num_results=num_results,
        )

    def build_context(self, search_results):
        """Convert retrieved lesson documents into LLM context."""
        sections = []

        for doc in search_results:
            section = (
                f"Filename: {doc['filename']}\n"
                f"Content:\n{doc['content']}"
            )
            sections.append(section)

        return "\n\n".join(sections)

    def build_prompt(self, query, search_results):
        """Build the final user prompt from the question and evidence."""
        context = self.build_context(search_results)

        return PROMPT_TEMPLATE.format(
            question=query,
            context=context,
        )

    def llm(self, prompt):
        """Call the Responses API and keep the full response object."""
        response = self.llm_client.responses.create(
            model=self.model,
            instructions=INSTRUCTIONS,
            input=prompt,
        )

        return response

    def rag(self, query):
        """Run retrieval and generation, returning answer and usage."""
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)

        return response.output_text, response.usage


lesson_rag = LessonRAG(
    index=index,
    llm_client=client,
)

print("Homework RAG helper ready.")
print("Model:", lesson_rag.model)

Homework RAG helper ready.
Model: gpt-5.4-mini


In [4]:
# Cell 04 | Run Q3 RAG and inspect token usage
# Purpose: Answer the homework query with full lesson pages as retrieval context.
# Key points: Measure the number of input tokens sent to the model.
# Execution: This cell sends one OpenAI API request and therefore incurs API usage.

query = "How does the agentic loop keep calling the model until it stops?"

answer, usage = lesson_rag.rag(query)

print("Answer:")
print(answer)

print("\nInput tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)
print("Total tokens:", usage.total_tokens)

Answer:
It keeps calling the model inside a `while True` loop. After each response, it checks whether the model returned any `function_call` items:

- if yes, it runs the tool, appends the tool result to `messages`, and loops again
- if no, it breaks

So the loop stops when a turn has no function calls, meaning the model has given its final answer.

Input tokens: 7135
Output tokens: 84
Total tokens: 7219


In [5]:
# Cell 05 | Chunk lesson documents
# Purpose: Split long lesson pages into smaller overlapping text windows.
# Key points: Use 2,000-character chunks with a 1,000-character step,
#             creating 1,000 characters of overlap between adjacent chunks.
# Execution: This is local preprocessing only. No OpenAI API request is made.

from gitsource import chunk_documents


chunks = chunk_documents(
    documents,
    size=2000,
    step=1000,
)

print("Original documents:", len(documents))
print("Total chunks:", len(chunks))

print("\nFirst chunk keys:", chunks[0].keys())
print("First chunk filename:", chunks[0]["filename"])
print("First chunk start:", chunks[0]["start"])
print("First chunk length:", len(chunks[0]["content"]))

Original documents: 72
Total chunks: 295

First chunk keys: dict_keys(['start', 'content', 'filename'])
First chunk filename: 01-agentic-rag/lessons/01-intro.md
First chunk start: 0
First chunk length: 2000


In [6]:
# Cell 06 | Build the chunked lesson search index
# Purpose: Index overlapping lesson chunks instead of full lesson pages.
# Key points: Keep the same MinSearch field configuration so Q5 changes
#             only document granularity, not the retrieval method.
# Execution: Local indexing and retrieval only. No OpenAI API request is made.

chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

chunk_index.fit(chunks)

chunk_results = chunk_index.search(
    query=query,
    num_results=5,
)

print("Indexed chunks:", len(chunks))

print("\nTop 5 chunk results:")
for rank, result in enumerate(chunk_results, start=1):
    print(
        f"{rank}. {result['filename']} "
        f"(start={result['start']}, length={len(result['content'])})"
    )

Indexed chunks: 295

Top 5 chunk results:
1. 01-agentic-rag/lessons/14-agentic-loop.md (start=4000, length=2000)
2. 01-agentic-rag/lessons/14-agentic-loop.md (start=5000, length=2000)
3. 01-agentic-rag/lessons/14-agentic-loop.md (start=0, length=2000)
4. 01-agentic-rag/lessons/15-frameworks.md (start=4000, length=1379)
5. 01-agentic-rag/lessons/15-frameworks.md (start=3000, length=2000)


In [7]:
# Cell 07 | Run Q5 chunked RAG and compare token usage
# Purpose: Measure how chunking changes the amount of context sent to the LLM.
# Key points: Reuse the same RAG helper, model, query, and Top-5 retrieval count.
#             Only the indexed document granularity changes from pages to chunks.
# Execution: This cell sends one OpenAI API request and incurs API usage.

full_page_input_tokens = usage.input_tokens

chunk_rag = LessonRAG(
    index=chunk_index,
    llm_client=client,
)

chunk_answer, chunk_usage = chunk_rag.rag(query)

reduction_factor = full_page_input_tokens / chunk_usage.input_tokens

print("Chunked RAG answer:")
print(chunk_answer)

print("\nFull-page input tokens:", full_page_input_tokens)
print("Chunked input tokens:", chunk_usage.input_tokens)
print("Chunked output tokens:", chunk_usage.output_tokens)
print("Chunked total tokens:", chunk_usage.total_tokens)
print(f"Reduction factor: {reduction_factor:.2f}x")

Chunked RAG answer:
The loop keeps calling the model inside a `while True` loop and checks whether the model produced any `function_call` items.

- If there is at least one function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: it repeats until the model returns a final answer with no more tool calls.

Full-page input tokens: 7135
Chunked input tokens: 2318
Chunked output tokens: 116
Chunked total tokens: 2434
Reduction factor: 3.08x


In [8]:
# Cell 08 | Build the Q6 ToyAIKit agent
# Purpose: Turn the chunked lesson search index into an agent tool.
# Key points: Let the model decide the search queries and how many searches
#             are needed before producing the final answer.
# Execution: This cell only configures the agent.
#            No OpenAI API request is made.

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner


def search(query: str):
    """Search course lesson chunks relevant to the student's question.

    Args:
        query: The search query used to retrieve relevant course content.

    Returns:
        The five most relevant lesson chunks.
    """
    return chunk_index.search(
        query=query,
        num_results=5,
    )


# Register the real Python search function as an LLM tool.
tools = Tools()
tools.add_tool(search)


AGENT_INSTRUCTIONS = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
""".strip()


# Reuse the existing OpenAI client and use the homework model.
agent_llm = OpenAIClient(
    model="gpt-5.4-mini",
    client=client,
)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=AGENT_INSTRUCTIONS,
    llm_client=agent_llm,
)


print("Q6 agent ready.")
print("Registered tools:", list(tools.functions.keys()))

Q6 agent ready.
Registered tools: ['search']


In [9]:
# Cell 09 | Run the Q6 agent and count search tool calls
# Purpose: Let the agent decide how many searches are needed before answering.
# Key points: Inspect the framework trajectory and count actual search calls
#             instead of inferring the number from the final answer.
# Execution: This cell sends OpenAI API requests through the agent loop
#            and therefore incurs API usage.

agent_question = (
    "How does the agentic loop work, "
    "and how is it different from plain RAG?"
)

agent_result = runner.loop(agent_question)

function_calls = [
    message
    for message in agent_result.new_messages
    if getattr(message, "type", None) == "function_call"
]

print("Final answer:")
print(agent_result.last_message)

print("\nSearch calls:", len(function_calls))

print("\nSearch queries:")
for number, call in enumerate(function_calls, start=1):
    print(f"{number}. {call.arguments}")

Final answer:
The **agentic loop** is the repeating cycle where the model can decide to use tools before answering.

### How it works
Typical flow:

1. You send the user question to the model.
2. The model may return a **tool/function call** instead of a final answer.
3. Your code executes that tool and sends the result back to the model.
4. The model can call more tools if needed.
5. The loop stops when the model returns a final answer with **no more tool calls**.

In the course, this is described as a `while` loop that:
- calls the LLM,
- executes any function calls it requested,
- appends the tool outputs to the message history,
- repeats until there are no more function calls.

### How it differs from plain RAG
**Plain RAG** is usually a simpler, fixed pipeline:

```python
search_results = search(question)
prompt = build_prompt(question, search_results)
answer = llm(prompt)
```

So the steps are predetermined:
- retrieve once,
- build one prompt,
- generate one answer.

### Key dif

In [10]:
# Cell 10 | Verify the Q6 search-call count
# Purpose: Confirm the exact number of search tool calls without rerunning the agent.
# Key points: Read the existing agent trajectory from Cell 09 and print only
#             compact verification data to avoid notebook output truncation.
# Execution: Run after Cell 09. No OpenAI API request is made.

function_calls = [
    message
    for message in agent_result.new_messages
    if getattr(message, "type", None) == "function_call"
]

print("Verified search calls:", len(function_calls))

for number, call in enumerate(function_calls, start=1):
    print(f"{number}. {call.name}: {call.arguments}")

Verified search calls: 4
1. search: {"query":"agentic loop plain RAG difference retrieval augmented generation loop agentic workflow"}
2. search: {"query":"agentic loop course lesson chunks RAG iterative tool use planning reflection"}
3. search: {"query":"plain RAG vs agentic loop course content"}
4. search: {"query":"agentic loop retrieval augmented generation differences search"}
